# Job Recommendation System — Clean Notebook

This notebook builds a hybrid job recommender using TF-IDF text similarity, skill coverage, and a technical-role relevance heuristic. Run cells from top to bottom.


## 1. Load data and build TF-IDF features


In [1]:

import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Load the cleaned dataset
df = pd.read_csv("../data/processed/jobs_cleaned.csv")

# Safely handle missing values and convert text columns to strings
text_columns = [
    "job_title",
    "category",
    "job_description",
    "job_skill_set"
]

for column in text_columns:
    df[column] = df[column].fillna("").astype(str)

# Give greater importance to titles, categories, and skills
df["recommendation_text"] = (
    (df["job_title"] + " ") * 2
    + (df["category"] + " ") * 2
    + (df["job_skill_set"] + " ") * 2
    + df["job_description"]
)

# Convert job text into numerical TF-IDF vectors
tfidf = TfidfVectorizer(
    stop_words="english",
    max_features=10000,
    ngram_range=(1, 2),
    sublinear_tf=True
)

tfidf_matrix = tfidf.fit_transform(df["recommendation_text"])

print("Dataset shape:", df.shape)
print("TF-IDF matrix shape:", tfidf_matrix.shape)
print("Vocabulary size:", len(tfidf.vocabulary_))

# Save the updated model components
project_dir = Path.cwd().parent
model_dir = project_dir / "models"
model_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(tfidf, model_dir / "tfidf_vectorizer.pkl")
joblib.dump(tfidf_matrix, model_dir / "tfidf_matrix.pkl")

print("Updated TF-IDF model saved successfully!")


Dataset shape: (1167, 7)
TF-IDF matrix shape: (1167, 10000)
Vocabulary size: 10000
Updated TF-IDF model saved successfully!


## 2. Skill parsing and normalization


In [2]:
import ast
import re

SKILL_ALIASES = {
    "python": {"python"},
    "sql": {"sql", "structured query language"},
    "pandas": {"pandas"},
    "scikit-learn": {"scikit-learn", "sklearn"},
    "machine learning": {"machine learning", "ml"},
}


def parse_job_skills(value):
    """Convert a dataset skill field into a list of lowercase skills."""
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return [str(skill).strip().lower() for skill in value]
    if isinstance(value, str):
        try:
            parsed = ast.literal_eval(value)
            if isinstance(parsed, list):
                return [str(skill).strip().lower() for skill in parsed]
        except (ValueError, SyntaxError):
            pass
        return [skill.strip().lower() for skill in value.split(",") if skill.strip()]
    return []


def normalize_skill(skill):
    skill = str(skill).lower().strip()
    for canonical, aliases in SKILL_ALIASES.items():
        if skill in aliases:
            return canonical
    return skill


def calculate_skill_coverage(user_input, job_skill_value):
    """Fraction of entered skills that are present in the job's skill list."""
    user_skills = {
        normalize_skill(skill)
        for skill in user_input.split(",")
        if skill.strip()
    }
    job_skills = {
        normalize_skill(skill)
        for skill in parse_job_skills(job_skill_value)
    }
    if not user_skills:
        return 0.0
    return len(user_skills & job_skills) / len(user_skills)


def get_matching_skills(user_input, job_skill_value):
    user_skills = {
        normalize_skill(skill)
        for skill in user_input.split(",")
        if skill.strip()
    }
    job_skills = {
        normalize_skill(skill)
        for skill in parse_job_skills(job_skill_value)
    }
    return sorted(user_skills & job_skills)


## 3. Technical-role relevance and hybrid recommender

The role score is a transparent heuristic, not a trained classifier. Pipeline/repository postings are excluded from recommendations.


In [3]:
def technical_role_relevance(job_title, job_description):
    title = str(job_title).lower().strip()
    description = str(job_description).lower()

    # Reject clearly non-technical roles.
    non_technical = [
        "business development representative",
        "sales representative",
        "account executive",
        "recruiter",
    ]

    if any(term in title for term in non_technical):
        return 0.0

    # Strong title matches for data, AI/ML, and software jobs.
    strong_titles = [
        "data scientist",
        "machine learning engineer",
        "ml engineer",
        "ai engineer",
        "data analyst",
        "data engineer",
        "software engineer",
        "software developer",
        "python developer",
        "backend developer",
        "full stack developer",
        "business intelligence analyst",
    ]

    if any(term in title for term in strong_titles):
        return 1.0

    # Other technical roles get a moderate score.
    moderate_titles = [
        "database administrator",
        "devops engineer",
        "cloud engineer",
        "information technology specialist",
        "it analyst",
        "systems analyst",
        "software architect",
    ]

    if any(term in title for term in moderate_titles):
        return 0.6

    # A description alone is weaker evidence than a title.
    technical_description_terms = [
        "software development",
        "data engineering",
        "machine learning",
        "programming",
        "data analytics",
    ]

    if any(term in description for term in technical_description_terms):
        return 0.3

    return 0.0


def recommend_jobs_hybrid_v2(user_input, top_n=10):
    if not isinstance(user_input, str) or not user_input.strip():
        raise ValueError("Please enter your skills.")
    if not isinstance(top_n, int) or top_n < 1:
        raise ValueError("top_n must be a positive integer.")

    user_vector = tfidf.transform([user_input])
    text_scores = cosine_similarity(user_vector, tfidf_matrix).flatten()

    skill_scores = np.array([
        calculate_skill_coverage(user_input, value)
        for value in df["job_skill_set"]
    ])

    title_scores = np.array([
        technical_role_relevance(title, description)
        for title, description in zip(df["job_title"], df["job_description"])
    ])

    final_scores = 0.35 * text_scores + 0.45 * skill_scores + 0.20 * title_scores

    descriptions = df["job_description"].fillna("").astype(str).str.lower()
    pipeline_phrases = [
        "does not represent an actual job opening",
        "pipeline/repository",
        "talent pool",
        "future opportunities",
    ]
    invalid_postings = descriptions.apply(
        lambda description: any(phrase in description for phrase in pipeline_phrases)
    ).to_numpy()

    # Only rank valid postings that have at least some text or skill match.
    eligible = (~invalid_postings) & ((text_scores > 0) | (skill_scores > 0))
    matching_indices = np.where(eligible)[0]
    ranked_indices = matching_indices[
        np.argsort(final_scores[matching_indices])[::-1]
    ][:top_n]

    results = df.iloc[ranked_indices][
        ["job_id", "job_title", "category", "job_skill_set", "job_description"]
    ].copy()
    results["job_row_id"] = ranked_indices
    results["matched_skills"] = results["job_skill_set"].apply(
        lambda value: get_matching_skills(user_input, value)
    )
    results["text_similarity"] = text_scores[ranked_indices]
    results["skill_coverage"] = skill_scores[ranked_indices]
    results["title_relevance"] = title_scores[ranked_indices]
    results["final_score"] = final_scores[ranked_indices]
    return results.reset_index(drop=True)


## 4. Try a user profile


In [4]:
query = "Python, machine learning, pandas, scikit-learn, SQL"
results = recommend_jobs_hybrid_v2(query, top_n=10)

display(
    results[[
        "job_title", "category", "matched_skills",
        "text_similarity", "skill_coverage", "title_relevance", "final_score"
    ]].round(3)
)


,job_title,category,matched_skills,text_similarity,skill_coverage,title_relevance,final_score
0,Lead Software Engineer - Finance (Hybrid),FINANCE,"[machine learning, python, sql]",0.248,0.6,1.0,0.557
1,"VP / Client, Finance & Strategy Data Scientist...",FINANCE,"[machine learning, python, sql]",0.196,0.6,1.0,0.538
2,Staff Data Engineer - People & Finance Tech (R...,FINANCE,"[python, sql]",0.105,0.4,1.0,0.417
3,"Data Engineer - Enterprise Data and Analytics,...",HR,"[python, sql]",0.086,0.4,1.0,0.410
4,Information Technology Specialist,INFORMATION-TECHNOLOGY,"[python, sql]",0.149,0.4,0.6,0.352
5,Senior Oracle Database Administrator (Senior I...,INFORMATION-TECHNOLOGY,[sql],0.089,0.2,0.6,0.241
6,Information Technology Specialist II,INFORMATION-TECHNOLOGY,[sql],0.064,0.2,0.6,0.232
7,Information Technology Specialist 2 (NY HELPs)...,INFORMATION-TECHNOLOGY,[sql],0.034,0.2,0.6,0.222
8,Data Solutions Analyst II (Managed Care - Fina...,FINANCE,"[python, sql]",0.097,0.4,0.0,0.214
9,Analyst II - Information Technology,INFORMATION-TECHNOLOGY,[machine learning],0.139,0.2,0.3,0.199


## 5. Inspect recommended job details


In [5]:
for _, row in results.iterrows():
    print("=" * 90)
    print("Row position:", int(row["job_row_id"]))
    print("Job ID:", row["job_id"])
    print("Title:", row["job_title"])
    print("Category:", row["category"])
    print("Matched skills:", row["matched_skills"])
    print("Skill coverage:", round(float(row["skill_coverage"]), 3))
    print("Final score:", round(float(row["final_score"]), 3))
    print("Description:", str(row["job_description"])[:1200], "...")


Row position: 897
Job ID: 3906246896
Title: Lead Software Engineer - Finance (Hybrid)
Category: FINANCE
Matched skills: ['machine learning', 'python', 'sql']
Skill coverage: 0.6
Final score: 0.557
Description: JOB DESCRIPTION

AS A VALUED COLLEAGUE ON OUR TEAM, YOU WILL ACT AS A TEAM LEAD IN THE DESIGNING, PRODUCING, TESTING, OR IMPLEMENTING SOFTWARE, TECHNOLOGY, OR PROCESSES, AS WELL AS LEAD PROCESSES FOR CREATING AND MAINTAINING IT ARCHITECTURE, LARGE SCALE DATA STORES, AND CLOUD-BASED SYSTEMS.

THE IMPACT YOU WILL MAKE

RESPONSIBILITIES

THE LEAD SOFTWARE ENGINEER - FINANCE ROLE WILL OFFER YOU THE FLEXIBILITY TO MAKE EACH DAY YOUR OWN, WHILE WORKING ALONGSIDE PEOPLE WHO CARE SO THAT YOU CAN DELIVER ON THE FOLLOWING RESPONSIBILITIES:

INDEPENDENTLY DETERMINE THE NEEDS OF THE CUSTOMER WHILE IDENTIFYING AND RESOLVING CONFLICTING OR COMPLEMENTARY NEEDS ACROSS CUSTOMER GROUPS.APPLYING ADVANCED SKILL, KNOWLEDGE AND EXPERIENCE, DESIGN AND DEVELOP SOFTWARE SOLUTIONS TO MEET CUSTOMER NEEDS.U

## 6. Flag seniority for manual review

This does not remove senior jobs automatically; it only flags titles that may be too senior for an entry-level candidate.


In [6]:
SENIORITY_TERMS = [
    "senior", "staff", "lead", "principal", "director",
    "vice president", "vp", "head of", "manager", "chief",
]

def detect_seniority(job_title):
    title = str(job_title).lower()
    pattern = r"\b(?:" + "|".join(re.escape(term) for term in SENIORITY_TERMS) + r")\b"
    return "Potentially senior" if re.search(pattern, title) else "Seniority not indicated"

review = results[["job_title", "final_score", "skill_coverage"]].copy()
review["seniority_flag"] = review["job_title"].apply(detect_seniority)
display(review)


,job_title,final_score,skill_coverage,seniority_flag
0,Lead Software Engineer - Finance (Hybrid),0.556702,0.6,Potentially senior
1,"VP / Client, Finance & Strategy Data Scientist...",0.538489,0.6,Potentially senior
2,Staff Data Engineer - People & Finance Tech (R...,0.416811,0.4,Potentially senior
3,"Data Engineer - Enterprise Data and Analytics,...",0.409997,0.4,Seniority not indicated
4,Information Technology Specialist,0.352067,0.4,Seniority not indicated
5,Senior Oracle Database Administrator (Senior I...,0.241047,0.2,Potentially senior
6,Information Technology Specialist II,0.232257,0.2,Seniority not indicated
7,Information Technology Specialist 2 (NY HELPs)...,0.221846,0.2,Seniority not indicated
8,Data Solutions Analyst II (Managed Care - Fina...,0.213867,0.4,Seniority not indicated
9,Analyst II - Information Technology,0.198644,0.2,Seniority not indicated


## Notes

- The scoring weights are hand-chosen heuristics and should be tuned using genuine manually reviewed relevance labels.
- Do not report Precision@5 from placeholder labels. Create labels from your own review before evaluating.
- The TF-IDF model artifacts are saved to the project `models` folder when the setup cell runs.
